<a href="https://colab.research.google.com/github/pavlolubtsov-lab/lab2/blob/main/HTMLgame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Градостроитель. Игровая логика работает на Python, интерфейс — в браузере."""

from __future__ import annotations

import json
import random
import threading
import time
import webbrowser
from dataclasses import asdict, dataclass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer


@dataclass(frozen=True)
class Level:
    name: str
    width: int
    height: int
    capacity: int
    money: int
    materials: int


@dataclass
class Building:
    kind: str
    level: int
    x: int
    y: int
    width: int
    height: int
    workers: int = 0
    connected: bool = False
    construction_ticks: int = 0
    fire_ticks: int = 0


DATA = {
    "house": {"name": "Дом", "color": "#d9a066", "levels": [
        Level("Частный дом", 1, 1, 5, 100, 10), Level("Двухэтажный дом", 1, 1, 10, 250, 30),
        Level("Малый жилой комплекс", 2, 1, 20, 600, 80), Level("Большой жилой комплекс", 2, 1, 40, 1400, 200)]},
    "factory": {"name": "Завод", "color": "#8a93a8", "levels": [
        Level("Малое предприятие", 1, 1, 10, 200, 20), Level("Небольшое предприятие", 2, 1, 20, 500, 60),
        Level("Среднее предприятие", 2, 1, 40, 1200, 150), Level("Большой завод", 2, 2, 80, 3000, 400),
        Level("Крупный промкомбинат", 2, 2, 160, 7000, 1000)]},
    "office": {"name": "Офис", "color": "#5aa9d6", "levels": [
        Level("Малый бизнес", 1, 1, 5, 150, 15), Level("Небольшая фирма", 1, 1, 10, 400, 40),
        Level("Малое агентство", 1, 1, 20, 1000, 100), Level("Среднее агентство", 1, 1, 40, 2500, 250),
        Level("Крупная корпорация", 2, 1, 80, 6000, 600)]},
    "school": {"name": "Школа", "color": "#e6c85c", "levels": [
        Level("Начальная школа", 1, 1, 0, 300, 40), Level("Средняя школа", 1, 1, 0, 700, 100),
        Level("Большая школа", 1, 1, 0, 1500, 220)]},
    "park": {"name": "Парк", "color": "#176b35", "levels": [Level("Парк", 1, 1, 0, 250, 30)]},
    "statue": {"name": "Статуя", "color": "#b8c2cc", "levels": [Level("Статуя", 1, 1, 0, 200, 20)]},
    "fountain": {"name": "Фонтан", "color": "#48b9d1", "levels": [Level("Фонтан", 1, 1, 0, 350, 35)]},
    "museum": {"name": "Музей", "color": "#b98a58", "levels": [Level("Музей", 2, 2, 0, 900, 120)]},
    "fire": {"name": "Пожарная служба", "color": "#dc5b4f", "levels": [
        Level("Пожарный пост", 1, 1, 5, 450, 60), Level("Пожарная часть", 2, 1, 15, 1200, 180),
        Level("Пожарный депо-комплекс", 2, 2, 30, 2800, 450)]},
}

SOCIAL_KINDS = {"school", "park", "statue", "fountain", "museum"}
LANDMARK_KINDS = {"statue", "fountain", "museum"}
SCHOOL_BONUS = (5, 10, 15)
LANDMARK_BONUS = {"statue": 10, "fountain": 12, "museum": 15}
SOCIAL_RADIUS = 5
FIRE_RADIUS = (3, 6, 10)
FIRE_CHANCE = 0.01


class City:
    def __init__(self) -> None:
        self.lock = threading.RLock()
        self.money, self.materials, self.population = 1000, 100, 5
        self.speed, self.elapsed, self.unemployed_ticks = 1, 0.0, 0
        self.roads = {(x, 5) for x in range(1, 9)}
        self.plots = {(0, 0)}
        self.buildings = [Building("house", 0, 2, 4, 1, 1), Building("factory", 0, 4, 4, 1, 1),
                          Building("office", 0, 6, 6, 1, 1)]
        self.network: set[tuple[int, int]] = set()
        self.message = ""
        self.message_until = 0.0
        self.capacity = self.employed = self.jobs = 0
        self.happiness_decay = 0
        self.house_happiness: dict[int, int] = {}
        self.happiness = 10.0
        self.recalculate()

    @staticmethod
    def plot(x: int, y: int) -> tuple[int, int]:
        return x // 10, y // 10

    def plot_cost(self) -> int:
        return 300 * len(self.plots) + 100

    def notify(self, message: str) -> None:
        self.message, self.message_until = message, time.monotonic() + 4

    def building_at(self, x: int, y: int) -> Building | None:
        return next((b for b in self.buildings if b.kind in LANDMARK_KINDS and
                     b.x <= x < b.x + b.width and b.y <= y < b.y + b.height), None) or next(
                     (b for b in self.buildings if b.kind != "park" and
                      b.x <= x < b.x + b.width and b.y <= y < b.y + b.height), None) or next(
                     (b for b in self.buildings if b.kind == "park" and
                      b.x <= x < b.x + b.width and b.y <= y < b.y + b.height), None)

    def park_at(self, x: int, y: int) -> Building | None:
        return next((b for b in self.buildings if b.kind == "park" and b.x <= x < b.x + b.width
                     and b.y <= y < b.y + b.height), None)

    def fire_protects(self, target: Building) -> bool:
        for station in self.buildings:
            if station.kind != "fire" or not station.connected or station.construction_ticks > 0:
                continue
            required = DATA["fire"]["levels"][station.level].capacity
            if station.workers < required:
                continue
            if self.near(target, station, FIRE_RADIUS[station.level]):
                return True
        return False

    @staticmethod
    def near(a: Building, b: Building, radius: int = SOCIAL_RADIUS) -> bool:
        ax, ay = a.x + a.width / 2, a.y + a.height / 2
        bx, by = b.x + b.width / 2, b.y + b.height / 2
        return (ax - bx) ** 2 + (ay - by) ** 2 <= radius ** 2

    def recalculate_happiness(self) -> None:
        homes = [b for b in self.buildings if b.kind == "house" and b.construction_ticks == 0]
        values = {}
        for home in homes:
            value = 10
            for amenity in self.buildings:
                if not amenity.connected or amenity.construction_ticks > 0 or not self.near(home, amenity):
                    continue
                if amenity.kind == "school":
                    value += SCHOOL_BONUS[amenity.level]
                elif amenity.kind == "park":
                    value += 5
                elif amenity.kind in LANDMARK_BONUS:
                    value += LANDMARK_BONUS[amenity.kind]
            values[id(home)] = max(0, value - self.happiness_decay)
        self.house_happiness = values
        self.happiness = sum(values.values()) / len(values) if values else 0.0

    def recalculate(self) -> None:
        remaining = set(self.roads)
        largest: set[tuple[int, int]] = set()
        while remaining:
            start = remaining.pop()
            component, todo = {start}, [start]
            while todo:
                x, y = todo.pop()
                for neighbor in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                    if neighbor in remaining:
                        remaining.remove(neighbor)
                        component.add(neighbor)
                        todo.append(neighbor)
            if len(component) > len(largest):
                largest = component
        self.network = largest
        for b in self.buildings:
            b.connected = any((b.x + dx, b.y + dy) in largest
                              for dx in range(-1, b.width + 1) for dy in range(-1, b.height + 1)
                              if (dx < 0 or dx >= b.width) != (dy < 0 or dy >= b.height))
        for b in self.buildings:
            if b.kind in LANDMARK_KINDS:
                park = self.park_at(b.x + b.width // 2, b.y + b.height // 2)
                b.connected = bool(park and park.connected)
        self.assign_workers()
        self.recalculate_happiness()

    def assign_workers(self) -> None:
        self.capacity = 0
        workplaces = []
        for b in self.buildings:
            b.workers = 0
            if not b.connected or b.construction_ticks > 0:
                continue
            level = DATA[b.kind]["levels"][b.level]
            if b.kind == "house":
                self.capacity += level.capacity
            elif b.kind in SOCIAL_KINDS:
                continue
            elif b.kind in {"factory", "office", "fire"}:
                workplaces.append(b)
        workplaces = [b for b in workplaces if not (b.kind in {"factory", "office"} and b.fire_ticks > 0)]
        workplaces.sort(key=lambda b: b.kind != "fire")
        self.population = min(self.population, self.capacity)
        self.jobs = sum(DATA[b.kind]["levels"][b.level].capacity for b in workplaces)
        available = self.population
        for b in workplaces:
            b.workers = min(available, DATA[b.kind]["levels"][b.level].capacity)
            available -= b.workers
        self.employed = self.population - available

    def tick(self) -> None:
        self.assign_workers()
        for b in self.buildings:
            if b.kind not in {"house", "factory", "office"} or b.construction_ticks > 0:
                continue
            if b.fire_ticks > 0:
                if self.fire_protects(b):
                    b.fire_ticks = 0
                    self.notify(f"Пожар ликвидирован пожарной службой: {DATA[b.kind]['name']}")
            elif random.random() < FIRE_CHANCE:
                if self.fire_protects(b):
                    self.notify(f"Пожар предотвращён пожарной службой: {DATA[b.kind]['name']}")
                else:
                    b.fire_ticks = random.randint(2, 3)
                    self.notify(f"Пожар! {DATA[b.kind]['name']} остановлено на {b.fire_ticks} тика")
        self.assign_workers()
        if not any(b.kind in SOCIAL_KINDS and b.connected and b.construction_ticks == 0 for b in self.buildings):
            self.happiness_decay += 1
        self.recalculate_happiness()
        severe = False
        for b in self.buildings:
            if b.kind == "house" or b.kind in SOCIAL_KINDS or not b.connected or b.construction_ticks > 0 or b.fire_ticks > 0:
                continue
            level = DATA[b.kind]["levels"][b.level]
            if b.workers / level.capacity < .2:
                severe = True
            elif b.kind == "factory":
                self.materials += b.workers * 2
            elif b.kind == "office":
                self.money += b.workers * 10
        vacant = self.capacity - self.population
        if vacant > 0:
            arrival_factor = 1.5 if self.happiness > 50 else 1
            self.population += min(vacant, int(random.random() * 21 * (.7 if severe else 1) * arrival_factor))
        self.assign_workers()
        if self.population > self.employed:
            self.unemployed_ticks += 1
            if self.unemployed_ticks >= 2:
                rate = 2 if self.happiness < 30 else 1
                left = min(self.population, (self.population - self.employed + 4) // 5 * rate)
                self.population -= left
                self.notify(f"Безработные покинули город: {left}")
        else:
            self.unemployed_ticks = 0
        for b in self.buildings:
            if b.construction_ticks > 0:
                b.construction_ticks -= 1
            if b.fire_ticks > 0:
                b.fire_ticks -= 1
        self.assign_workers()
        self.recalculate_happiness()

    def act(self, x: int, y: int, tool: str, paint: bool = False) -> None:
        plot = self.plot(x, y)
        if plot not in self.plots:
            if paint or not any((plot[0] + dx, plot[1] + dy) in self.plots for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1))):
                return
            cost = self.plot_cost()
            if self.money < cost:
                self.notify(f"Нужно {cost}$ для покупки участка")
                return
            self.money -= cost
            self.plots.add(plot)
            self.notify("Участок куплен!")
            return
        build_kind, build_level = tool, 0
        if ":" in tool:
            build_kind, raw_level = tool.split(":", 1)
            try:
                build_level = int(raw_level)
            except ValueError:
                return
            if build_kind not in DATA or not 0 <= build_level < len(DATA[build_kind]["levels"]):
                return
        b = self.building_at(x, y)
        if tool == "road":
            if b or (x, y) in self.roads or self.money < 10:
                return
            self.money -= 10
            self.roads.add((x, y))
            self.recalculate()
        elif tool == "demolish":
            if (x, y) in self.roads:
                self.roads.remove((x, y))
                self.money += 5
                self.recalculate()
            elif b:
                level = DATA[b.kind]["levels"][b.level]
                self.money += level.money // 2
                self.materials += level.materials // 2
                if b.kind == "park":
                    self.buildings = [other for other in self.buildings if not (
                        other.kind in LANDMARK_KINDS and b.x < other.x + other.width
                        and b.x + b.width > other.x and b.y < other.y + other.height
                        and b.y + b.height > other.y)]
                self.buildings.remove(b)
                self.recalculate()
        elif tool == "upgrade" and b:
            self.upgrade(b)
        elif build_kind in LANDMARK_KINDS:
            level = DATA[build_kind]["levels"][build_level]
            if any(not self.park_at(cx, cy) for cx in range(x, x + level.width) for cy in range(y, y + level.height)):
                self.notify("Достопримечательности можно строить только в парке")
                return
            if any(any(other.kind in LANDMARK_KINDS and other.x <= cx < other.x + other.width
                       and other.y <= cy < other.y + other.height for other in self.buildings)
                   for cx in range(x, x + level.width) for cy in range(y, y + level.height)):
                self.notify("Для достопримечательности нужно свободное место в парке")
                return
            if self.money < level.money or self.materials < level.materials:
                self.notify(f"Нужно {level.money}$ и {level.materials} стройматериалов")
                return
            self.money -= level.money
            self.materials -= level.materials
            self.buildings.append(Building(build_kind, build_level, x, y, level.width, level.height, construction_ticks=1))
            self.recalculate()
        elif build_kind in DATA and not b and (x, y) not in self.roads:
            level = DATA[build_kind]["levels"][build_level]
            if build_kind == "park" and any(self.plot(cx, cy) not in self.plots or (cx, cy) in self.roads or
                    any(other.x <= cx < other.x + other.width and other.y <= cy < other.y + other.height
                        for other in self.buildings) for cx in range(x, x + level.width) for cy in range(y, y + level.height)):
                self.notify("Для парка нужно свободное место на купленном участке")
                return
            if self.money < level.money or self.materials < level.materials:
                self.notify(f"Нужно {level.money}$ и {level.materials} стройматериалов")
                return
            self.money -= level.money
            self.materials -= level.materials
            self.buildings.append(Building(build_kind, build_level, x, y, level.width, level.height, construction_ticks=1))
            self.recalculate()

    def upgrade(self, b: Building) -> None:
        levels = DATA[b.kind]["levels"]
        index = b.level + 1
        if index >= len(levels):
            self.notify("Максимальный уровень")
            return
        level = levels[index]
        if self.money < level.money or self.materials < level.materials:
            self.notify(f"Нужно {level.money}$ и {level.materials} стройматериалов")
            return
        sizes = {(level.width, level.height), (level.height, level.width)}
        for width, height in sizes:
            for x in range(b.x - width + 1, b.x + b.width):
                for y in range(b.y - height + 1, b.y + b.height):
                    cells = {(x + dx, y + dy) for dx in range(width) for dy in range(height)}
                    old = {(b.x + dx, b.y + dy) for dx in range(b.width) for dy in range(b.height)}
                    if not old <= cells:
                        continue
                    collision = any(self.plot(cx, cy) not in self.plots or
                                    ((cx, cy) not in old and (cx, cy) in self.roads) or
                                    any(other is not b and other.x <= cx < other.x + other.width and
                                        other.y <= cy < other.y + other.height for other in self.buildings)
                                    for cx, cy in cells)
                    if collision:
                        continue
                    self.buildings.remove(b)
                    self.buildings.append(Building(b.kind, index, x, y, width, height, construction_ticks=1))
                    self.money -= level.money
                    self.materials -= level.materials
                    self.recalculate()
                    self.notify("Улучшено: " + level.name)
                    return
        self.notify("Нет свободного места для расширения")

    def snapshot(self) -> dict:
        with self.lock:
            self.assign_workers()
            return {
                "money": self.money, "materials": self.materials, "population": self.population,
                "happiness": round(self.happiness, 1),
                "employed": self.employed, "capacity": self.capacity, "jobs": self.jobs,
                "speed": self.speed, "remaining": max(0, int(60 - self.elapsed)),
                "plot_cost": self.plot_cost(), "plots": list(self.plots),
                "build_specs": {kind: {"color": data["color"], "levels": [
                    {"title": level.name, "width": level.width, "height": level.height,
                     "capacity": level.capacity, "cost": level.money, "materials": level.materials}
                    for level in data["levels"]]} for kind, data in DATA.items()},
                "roads": list(self.roads), "network": list(self.network),
                "buildings": [{**asdict(b), "title": DATA[b.kind]["levels"][b.level].name,
                                "limit": DATA[b.kind]["levels"][b.level].capacity,
                                "color": DATA[b.kind]["color"], "max_level": len(DATA[b.kind]["levels"]),
                                "happiness": self.house_happiness.get(id(b), 0),
                                "bonus": (SCHOOL_BONUS[b.level] if b.kind == "school" else
                                          LANDMARK_BONUS.get(b.kind, 5 if b.kind == "park" else 0)),
                                "radius": SOCIAL_RADIUS if b.kind in SOCIAL_KINDS and b.construction_ticks == 0 else 0,
                                "fire_radius": FIRE_RADIUS[b.level] if b.kind == "fire" else 0,
                                "fire_protected": self.fire_protects(b) if b.kind in {"house", "factory", "office"} else False}
                               for b in self.buildings],
                "message": self.message if time.monotonic() < self.message_until else "",
            }


GAME = City()

PAGE = r'''<!doctype html><html lang="ru"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Градостроитель</title><style>
*{box-sizing:border-box}body{margin:0;background:#15201a;color:#e8efe9;font:14px system-ui;overflow:hidden}header,footer{background:#1f2c24;display:flex;align-items:center;gap:8px;padding:7px;flex-wrap:wrap}header{flex-wrap:nowrap;overflow-x:auto;white-space:nowrap}header span{margin-right:12px}b{color:#5fc27a}canvas{display:block;width:100vw;height:calc(100vh - 205px);touch-action:none}footer{position:absolute;bottom:0;width:100%;height:170px;display:flex;flex-direction:column;align-items:stretch;gap:5px;padding:6px;overflow:hidden}button{background:#2c3d33;color:#e8efe9;border:1px solid #3d5546;border-radius:5px;padding:6px 9px;font:inherit;white-space:nowrap}button.on{background:#5fc27a;color:#0d1a11}#info{position:absolute;bottom:176px;left:10px;padding:5px 9px;background:#000a;border-radius:5px;pointer-events:none}.speeds{margin-left:auto;white-space:nowrap}#category-tabs,#menu-actions{display:flex;align-items:center;gap:5px;flex:none}#category-tabs button{padding:4px 8px}#building-cards{display:flex;align-items:stretch;gap:6px;min-height:77px;flex:1;overflow-x:auto;overflow-y:hidden}button.build-card{min-width:155px;max-width:210px;display:flex;flex-direction:column;align-items:flex-start;justify-content:center;gap:2px;padding:5px 8px;text-align:left;line-height:1.15}button.build-card .card-title{font-weight:700}button.build-card .card-cost{font-size:11px;color:#b9c9bd}button.build-card.unavailable{background:#633b3b;color:#d5bcbc;border-color:#8b4c4c}#menu-tooltip{display:none;position:fixed;z-index:2000;max-width:280px;padding:8px 10px;background:#101914;border:1px solid #6c8c74;border-radius:6px;box-shadow:0 4px 16px #000a;font-size:12px;line-height:1.4;white-space:pre-line;pointer-events:none}@media(max-width:650px){header{font-size:11px;gap:3px}header span{margin-right:3px}footer{height:180px;gap:3px;padding:4px}button{padding:5px;font-size:12px}canvas{height:calc(100vh - 215px)}#category-tabs{overflow-x:auto}#category-tabs button{font-size:11px;padding:4px 6px}button.build-card{min-width:142px}}
#tut-focus{display:none;position:fixed;z-index:1000;pointer-events:none;border:3px solid #ffe45c;border-radius:8px;box-shadow:0 0 0 100vmax rgba(0,0,0,.62);animation:tutPulse 1.1s ease-in-out infinite alternate}#tutorial{display:none;position:fixed;z-index:1002;left:50%;bottom:180px;transform:translateX(-50%);width:min(560px,calc(100vw - 24px));padding:16px 18px;background:#202e25;border:1px solid #ffe45c;border-radius:12px;box-shadow:0 8px 30px #000a}#tutorial h2{font-size:16px;margin:0 0 8px;color:#ffe45c}#tutorial p{line-height:1.45;margin:0 0 14px}#tutorial-actions{display:flex;justify-content:flex-end;gap:8px}#tutorial-skip{position:absolute;right:12px;top:10px;background:transparent;border-color:#718075}#tutorial-next{background:#5fc27a;color:#0d1a11;font-weight:700}@keyframes tutPulse{from{border-color:#ffe45c;opacity:.75}to{border-color:#fff;opacity:1}}
</style><header><span>💰 <b id="money"></b></span><span>🧱 <b id="materials"></b></span><span>👥 <b id="people"></b></span><span>💼 <b id="jobs"></b></span><span><b id="labor-status"></b></span><span>😊 Счастье <b id="happiness"></b>%</span><span>⏱ тик через <b id="time"></b> с</span></header>
<canvas id="game"></canvas><div id="info">ЛКМ — действие, перетаскивание — двигать карту, ПКМ — двигать карту</div>
<footer id="build-menu"><div id="category-tabs"><button class="on" data-category="Infrastructure">Инфраструктура</button><button data-category="Residential">Жильё</button><button data-category="Industry">Промышленность</button><button data-category="Commercial">Коммерция</button><button data-category="Services">Сервисы</button><button data-category="Safety">Безопасность</button></div><div id="building-cards"></div><div id="menu-actions"><button data-tool="upgrade">⬆ Улучшить</button><button data-tool="demolish">🗑 Снести</button><div class="speeds"><button data-speed="1">x1</button> <button data-speed="10">x10</button> <button data-speed="60">x60</button></div></div></footer><div id="menu-tooltip"></div>
<div id="tut-focus"></div><section id="tutorial" role="dialog" aria-modal="true"><button id="tutorial-skip">Пропустить обучение</button><h2 id="tutorial-title"></h2><p id="tutorial-text"></p><div id="tutorial-actions"><button id="tutorial-next"></button></div></section>
<script>
const canvas=document.querySelector('#game'),ctx=canvas.getContext('2d'),cell=32,plotSize=10;
let state=null,tool='',speed=1,camX=0,camY=0,hover=null,drag=null,initialized=false,tutorialSchoolNeedMove=false;
const tutorialKey='city-builder-tutorial',tutorialSteps=[
['Шаг 1 из 6 · Знакомство','Добро пожаловать в город! Сейчас выбран пустой курсор. Вы можете кликать по объектам, чтобы просматривать их статус. Нажмите ПКМ в любой момент, чтобы сбросить постройку.'],
['Шаг 2 из 6 · Дороги','Здания работают только при подключении к дороге! Выберите 🛣️ Дорогу в меню строительства и постройте ветку от базовых зданий.'],
['Шаг 3 из 6 · Жильё','Городу нужны жители! Постройте Частный дом (🏠). Обратите внимание: сначала появится иконка 🏗️ — постройка займёт 1 тик (1 минуту).'],
['Шаг 4 из 6 · Работа','Жителям нужна работа, а городу — стройматериалы и деньги. Следите за виджетом 💼 «Рабочие места» на верхней панели. Если на заводах и в офисах будет мало людей, они перестанут приносить ресурсы!'],
['Шаг 5 из 6 · Счастье','Чтобы жители не уезжали, стройте Школы (🏫) и Парки (🌲). Попробуйте выбрать Школу и наведите на карту: жёлтая прозрачная область покажет её радиус счастья!'],
['Шаг 6 из 6 · Расширение','Когда начнёт не хватать места, вы сможете покупать соседние участки 10×10 за деньги. Удачи в строительстве!']
];
const savedTutorial=document.cookie.split('; ').find(row=>row.startsWith(tutorialKey+'='))?.split('=')[1];
let tutorialStep=savedTutorial==='done'||localStorage.getItem(tutorialKey)==='done'?-1:Number(localStorage.getItem(tutorialKey)||0);
const key=(x,y)=>`${x},${y}`,plot=(x,y)=>`${Math.floor(x/10)},${Math.floor(y/10)}`;
function resize(){const d=devicePixelRatio||1,r=canvas.getBoundingClientRect();canvas.width=r.width*d;canvas.height=r.height*d;ctx.setTransform(d,0,0,d,0,0);if(!initialized){camX=r.width/2-5*cell;camY=r.height/2-5*cell-20;initialized=true}draw();updateTutorialFocus()}
window.addEventListener('resize',resize);
function clearTool(){tool='';document.querySelectorAll('[data-tool]').forEach(x=>x.classList.remove('on'));canvas.style.cursor='default';draw();updateTutorialFocus()}
function tutorialFocusBox(rect){const focus=document.querySelector('#tut-focus');if(!rect){focus.style.display='none';return}focus.style.display='block';focus.style.left=`${rect.left}px`;focus.style.top=`${rect.top}px`;focus.style.width=`${Math.max(8,rect.width)}px`;focus.style.height=`${Math.max(8,rect.height)}px`}
function updateTutorialFocus(){if(tutorialStep<0)return tutorialFocusBox(null);let el=null,world=null;if(tutorialStep===0)world=[2,4];else if(tutorialStep===1)toolKind()==='road'?world=[2,6]:el=document.querySelector('[data-tool="road"]');else if(tutorialStep===2)toolKind()==='house'?world=[3,6]:el=document.querySelector('[data-tool="house:0"]');else if(tutorialStep===3)el=document.querySelector('#jobs')?.closest('span')||document.querySelector('#jobs');else if(tutorialStep===4)toolKind()==='school'?world=[2,4]:el=document.querySelector('[data-tool="school:0"]');else if(tutorialStep===5)world=[10,0];if(world){const r=canvas.getBoundingClientRect();tutorialFocusBox({left:r.left+camX+world[0]*cell,top:r.top+camY+world[1]*cell,width:cell,height:cell})}else if(el){const r=el.getBoundingClientRect();tutorialFocusBox(r)}else tutorialFocusBox(null)}
function renderTutorial(){const card=document.querySelector('#tutorial');if(tutorialStep<0){card.style.display='none';tutorialFocusBox(null);return}card.style.display='block';document.querySelector('#tutorial-title').textContent=tutorialSteps[tutorialStep][0];document.querySelector('#tutorial-text').textContent=tutorialSteps[tutorialStep][1];const next=document.querySelector('#tutorial-next'),actions=document.querySelector('#tutorial-actions'),needsButton=[0,3,5].includes(tutorialStep);actions.style.display=needsButton?'flex':'none';next.textContent=tutorialStep===0?'Понятно':tutorialStep===5?'Начать игру':'Далее';updateTutorialFocus()}
function setTutorialStep(step){tutorialStep=step;localStorage.setItem(tutorialKey,String(step));if(step===1){setBuildCategory('Infrastructure');if(state)localStorage.setItem('city-builder-tutorial-roads',String(state.roads.length))}if(step===2){setBuildCategory('Residential');if(state)localStorage.setItem('city-builder-tutorial-homes',String(state.buildings.filter(b=>b.kind==='house').length))}if(step===4){setBuildCategory('Services');tutorialSchoolNeedMove=true}renderTutorial()}
function completeTutorial(){tutorialStep=-1;localStorage.setItem(tutorialKey,'done');document.cookie=tutorialKey+'=done; Max-Age=31536000; Path=/; SameSite=Lax';renderTutorial()}
function checkTutorialProgress(){if(tutorialStep===1&&state){let start=localStorage.getItem('city-builder-tutorial-roads');if(start===null){start=String(state.roads.length);localStorage.setItem('city-builder-tutorial-roads',start)}if(state.roads.length>Number(start))setTutorialStep(2)}else if(tutorialStep===2&&state){let start=localStorage.getItem('city-builder-tutorial-homes');if(start===null){start=String(state.buildings.filter(b=>b.kind==='house').length);localStorage.setItem('city-builder-tutorial-homes',start)}if(state.buildings.filter(b=>b.kind==='house').length>Number(start))setTutorialStep(3)}else if(tutorialStep===4&&state&&toolKind()==='school'&&!tutorialSchoolNeedMove&&hover&&state.buildings.some(b=>b.kind==='house'&&hover[0]>=b.x&&hover[0]<b.x+b.width&&hover[1]>=b.y&&hover[1]<b.y+b.height))setTutorialStep(5)}
document.querySelector('#tutorial-next').onclick=()=>{if(tutorialStep===0)setTutorialStep(1);else if(tutorialStep===3)setTutorialStep(4);else if(tutorialStep===5)completeTutorial()};
document.querySelector('#tutorial-skip').onclick=completeTutorial;
document.addEventListener('click',e=>{const b=e.target.closest('[data-tool]');if(!b)return;tool=b.dataset.tool;document.querySelectorAll('[data-tool]').forEach(x=>x.classList.toggle('on',x===b));canvas.style.cursor='crosshair';if(tutorialStep===4&&toolKind()==='school')tutorialSchoolNeedMove=true;draw();updateTutorialFocus()});
window.addEventListener('pointerdown',e=>{if(e.button===2)clearTool()});window.addEventListener('contextmenu',e=>e.preventDefault());
document.querySelectorAll('[data-speed]').forEach(b=>b.onclick=async()=>{speed=+b.dataset.speed;document.querySelectorAll('[data-speed]').forEach(x=>x.classList.toggle('on',x===b));await fetch('/speed',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({speed})})});
function coords(e){const r=canvas.getBoundingClientRect();return [Math.floor((e.clientX-r.left-camX)/cell),Math.floor((e.clientY-r.top-camY)/cell)]}
function send(x,y,paint=false){fetch('/act',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({x,y,tool,paint})})}
const emojis={road:['🛣️'],house:['🏚️','🏠','🏡','🏢'],factory:['🏚️','🏭','🏭','🏭','🏭'],office:['🏪','🏣','🏢','🏬','🏬'],fire:['🚒','🚒','🚒'],school:['🏫','🏫','🏫'],park:['🌲'],statue:['🗿'],fountain:['⛲'],museum:['🏛️']},socialTools=new Set(['school','park','statue','fountain','museum']),center=b=>[b.x+b.width/2,b.y+b.height/2];
let activeCategory='Infrastructure',menuSignature='';
const categoryKinds={Infrastructure:['road','park'],Residential:['house'],Industry:['factory'],Commercial:['office'],Services:['school','statue','fountain','museum'],Safety:['fire']};
const toolKind=()=>tool.split(':')[0],toolLevel=()=>Number(tool.split(':')[1]||0),selectedSpec=()=>{if(toolKind()==='road')return{title:'Дорога',width:1,height:1,capacity:0,cost:10,materials:0};const data=state?.build_specs?.[toolKind()];return data?.levels?.[toolLevel()]||null};
function setBuildCategory(category){activeCategory=category;document.querySelectorAll('[data-category]').forEach(b=>b.classList.toggle('on',b.dataset.category===category));menuSignature='';renderBuildMenu()}
function cardEffect(kind,level,spec){if(kind==='road')return'Подключает здания к дорожной сети';if(kind==='house')return`+${spec.capacity} жителей`;if(kind==='factory'||kind==='office')return`+${spec.capacity} рабочих мест`;if(kind==='school')return`+${[5,10,15][level]}% счастья · радиус 5 клеток`;if(kind==='park')return'+5% счастья · радиус 5 клеток';if(kind==='fire')return`Радиус защиты: ${[3,6,10][level]} клеток · ${spec.capacity} пожарных`;if(kind==='statue')return'+10% счастья · строить в парке';if(kind==='fountain')return'+12% счастья · строить в парке';if(kind==='museum')return'+15% счастья · размещать в парке';return''}
function renderBuildMenu(){if(!state)return;const signature=`${activeCategory}:${state.money}:${state.materials}`;if(signature===menuSignature)return;menuSignature=signature;const cards=document.querySelector('#building-cards'),tip=document.querySelector('#menu-tooltip');cards.replaceChildren();for(const kind of categoryKinds[activeCategory]){const levels=kind==='road'?[{title:'Дорога',width:1,height:1,capacity:0,cost:10,materials:0}]:state.build_specs[kind].levels;levels.forEach((spec,index)=>{const id=kind==='road'?'road':`${kind}:${index}`,emoji=emojis[kind]?.[index]||'🏢',effect=cardEffect(kind,index,spec),affordable=state.money>=spec.cost&&state.materials>=spec.materials,card=document.createElement('button'),title=document.createElement('span'),cost=document.createElement('span');card.className='build-card'+(affordable?'':' unavailable')+(tool===id?' on':'');card.dataset.tool=id;card.dataset.tip=`Размер: ${spec.width}×${spec.height} клет${spec.width*spec.height===1?'ка':'ки'}\nЭффект: ${effect}\nСтоимость: 💰 ${spec.cost} · 📦 ${spec.materials}`;title.className='card-title';title.textContent=`${emoji} ${spec.title}${kind==='road'?'':` · ур. ${index+1}`}`;cost.className='card-cost';cost.textContent=`💰 ${spec.cost} · 📦 ${spec.materials}`;card.append(title,cost);card.addEventListener('pointerenter',()=>{const r=card.getBoundingClientRect();tip.textContent=card.dataset.tip;tip.style.display='block';tip.style.left=`${Math.max(8,Math.min(innerWidth-288,r.left))}px`;tip.style.top='0px';tip.style.top=`${Math.max(8,r.top-tip.offsetHeight-8)}px`});card.addEventListener('pointerleave',()=>tip.style.display='none');cards.append(card)})}}
document.querySelectorAll('[data-category]').forEach(b=>b.onclick=()=>setBuildCategory(b.dataset.category));
function fits(x,y,w,h,kind){const cells=[];for(let dx=0;dx<w;dx++)for(let dy=0;dy<h;dy++)cells.push([x+dx,y+dy]);const owned=new Set(state.plots.map(p=>p.join(','))),roads=new Set(state.roads.map(p=>p.join(',')));if(cells.some(([cx,cy])=>!owned.has(plot(cx,cy))))return false;if(kind==='road')return !roads.has(key(x,y))&&!state.buildings.some(b=>x>=b.x&&x<b.x+b.width&&y>=b.y&&y<b.y+b.height);if(['statue','fountain','museum'].includes(kind))return cells.every(([cx,cy])=>state.buildings.some(b=>b.kind==='park'&&cx>=b.x&&cx<b.x+b.width&&cy>=b.y&&cy<b.y+b.height))&&!state.buildings.some(b=>['statue','fountain','museum'].includes(b.kind)&&cells.some(([cx,cy])=>cx>=b.x&&cx<b.x+b.width&&cy>=b.y&&cy<b.y+b.height));return cells.every(([cx,cy])=>!roads.has(key(cx,cy))&&!state.buildings.some(b=>cx>=b.x&&cx<b.x+b.width&&cy>=b.y&&cy<b.y+b.height))}
canvas.oncontextmenu=e=>e.preventDefault();
canvas.onpointerdown=e=>{canvas.setPointerCapture(e.pointerId);const [x,y]=coords(e);const owned=state?.plots.some(p=>p[0]===Math.floor(x/10)&&p[1]===Math.floor(y/10));drag={x:e.clientX,y:e.clientY,moved:false,paint:toolKind()==='road'&&e.button===0&&owned};if(drag.paint)send(x,y,true)};
canvas.onpointermove=e=>{hover=coords(e);if(tutorialStep===4&&toolKind()==='school')tutorialSchoolNeedMove=false;if(!drag){draw();updateTutorialFocus();return}const dx=e.clientX-drag.x,dy=e.clientY-drag.y;if(drag.paint){const [x,y]=coords(e);send(x,y,true);drag.moved=true}else if(drag.moved||Math.hypot(dx,dy)>6){drag.moved=true;camX+=dx;camY+=dy;drag.x=e.clientX;drag.y=e.clientY}hover=coords(e);draw();updateTutorialFocus()};
canvas.onpointerup=e=>{if(drag&&!drag.moved&&e.button===0&&tool){const [x,y]=coords(e);send(x,y)}drag=null};
function draw(){if(!state){renderTutorial();return}checkTutorialProgress();const d=devicePixelRatio||1,w=canvas.width/d,h=canvas.height/d;ctx.clearRect(0,0,w,h);ctx.fillStyle='#0f1612';ctx.fillRect(0,0,w,h);ctx.save();ctx.translate(camX,camY);
const owned=new Set(state.plots.map(p=>p.join(','))),roads=new Set(state.roads.map(p=>p.join(','))),network=new Set(state.network.map(p=>p.join(','))),buy=new Set();
for(const [px,py] of state.plots){for(const [dx,dy] of [[1,0],[-1,0],[0,1],[0,-1]])if(!owned.has(`${px+dx},${py+dy}`))buy.add(`${px+dx},${py+dy}`);ctx.fillStyle='#2c4a30';ctx.fillRect(px*10*cell,py*10*cell,10*cell,10*cell);ctx.strokeStyle='#ffffff18';for(let i=0;i<=10;i++){ctx.beginPath();ctx.moveTo((px*10+i)*cell,py*10*cell);ctx.lineTo((px*10+i)*cell,(py*10+10)*cell);ctx.moveTo(px*10*cell,(py*10+i)*cell);ctx.lineTo((px*10+10)*cell,(py*10+i)*cell);ctx.stroke()}}
for(const p of buy){const [px,py]=p.split(',').map(Number);if((px*10*cell+camX>w||((px+1)*10*cell+camX)<0)||(py*10*cell+camY>h||((py+1)*10*cell+camY)<0))continue;ctx.fillStyle='#1a241d';ctx.fillRect(px*10*cell,py*10*cell,10*cell,10*cell);ctx.strokeStyle='#5fc27a88';ctx.setLineDash([8,6]);ctx.strokeRect(px*10*cell+2,py*10*cell+2,10*cell-4,10*cell-4);ctx.setLineDash([]);ctx.fillStyle=state.money>=state.plot_cost?'#5fc27a':'#888';ctx.textAlign='center';ctx.font='bold 16px system-ui';ctx.fillText('+ Купить',(px*10+5)*cell,(py*10+5)*cell-4);ctx.fillText(state.plot_cost+'$',(px*10+5)*cell,(py*10+5)*cell+16)}
for(const p of roads){const [x,y]=p.split(',').map(Number);ctx.fillStyle=network.has(p)?'#555d66':'#7a5a5a';ctx.fillRect(x*cell,y*cell,cell,cell)}
let aura=null;if(hover){const [hx,hy]=hover;const under=state.buildings.find(b=>b.kind!=='park'&&hx>=b.x&&hx<b.x+b.width&&hy>=b.y&&hy<b.y+b.height)||state.buildings.find(b=>b.kind==='park'&&hx>=b.x&&hx<b.x+b.width&&hy>=b.y&&hy<b.y+b.height);if(under?.radius)aura={...under};else if(socialTools.has(toolKind())){const spec=selectedSpec();aura={x:hx,y:hy,width:spec.width,height:spec.height,radius:5,preview:true}}}if(aura){const [ax,ay]=center(aura);ctx.fillStyle='rgba(255,255,0,0.25)';ctx.beginPath();ctx.arc(ax*cell,ay*cell,aura.radius*cell,0,Math.PI*2);ctx.fill()}
let fireAura=null;if(hover){const [fx,fy]=hover,station=state.buildings.find(b=>b.kind==='fire'&&fx>=b.x&&fx<b.x+b.width&&fy>=b.y&&fy<b.y+b.height);if(station)fireAura={...station,radius:station.fire_radius};else if(toolKind()==='fire'){const spec=selectedSpec();fireAura={x:fx,y:fy,width:spec.width,height:spec.height,radius:[3,6,10][toolLevel()]}}}if(fireAura?.radius){const [fx,fy]=center(fireAura);ctx.fillStyle='rgba(255,0,0,0.2)';ctx.beginPath();ctx.arc(fx*cell,fy*cell,fireAura.radius*cell,0,Math.PI*2);ctx.fill()}
for(const b of state.buildings){const x=b.x*cell,y=b.y*cell,wid=b.width*cell,hei=b.height*cell,ratio=b.workers/Math.max(1,b.limit),emoji=b.construction_ticks>0?'🏗️':emojis[b.kind]?.[b.level]||'🏢';ctx.fillStyle=b.kind==='park'?'#145b2b':b.color;ctx.fillRect(x+2,y+2,wid-4,hei-4);if(b.kind==='park'){ctx.fillStyle='#176b35';for(let i=1;i<4;i++){ctx.fillRect(x+i*wid/4-2,y+2,4,hei-4);ctx.fillRect(x+2,y+i*hei/4-2,wid-4,4)}}if(['factory','office'].includes(b.kind)&&b.construction_ticks===0){ctx.fillStyle='#000a';ctx.fillRect(x+4,y+hei-9,wid-8,4);ctx.fillStyle=ratio<.2?'#e8883a':'#5fc27a';ctx.fillRect(x+4,y+hei-9,(wid-8)*ratio,4)}ctx.strokeStyle=!b.connected?'#ff4040':['factory','office'].includes(b.kind)&&ratio<.2?'#e8883a':'#273128';ctx.lineWidth=!b.connected?3:1;ctx.strokeRect(x+2,y+2,wid-4,hei-4);ctx.font=`${Math.min(wid,hei)*.68}px "Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif`;ctx.textAlign='center';ctx.textBaseline='middle';ctx.fillText(emoji,x+wid/2,y+hei/2)}
if(aura&&(aura.preview||aura.connected))for(const b of state.buildings.filter(b=>b.kind==='house'&&b.construction_ticks===0)){const [ax,ay]=center(aura),[bx,by]=center(b);if((ax-bx)**2+(ay-by)**2<=aura.radius**2){ctx.strokeStyle='rgba(255,255,120,0.9)';ctx.lineWidth=3;ctx.strokeRect(b.x*cell+2,b.y*cell+2,b.width*cell-4,b.height*cell-4)}}
if(hover&&!(drag?.moved&&!drag.paint)&&(toolKind()==='road'||selectedSpec())){const [gx,gy]=hover,spec=selectedSpec(),kind=toolKind(),level=toolLevel(),valid=state.money>=spec.cost&&state.materials>=spec.materials&&fits(gx,gy,spec.width,spec.height,kind);ctx.fillStyle=valid?'rgba(80,220,120,0.5)':'rgba(255,55,55,0.5)';ctx.fillRect(gx*cell+1,gy*cell+1,spec.width*cell-2,spec.height*cell-2);if(kind!=='road'){ctx.globalAlpha=.5;ctx.font=`${Math.min(spec.width,spec.height)*cell*.68}px "Apple Color Emoji","Segoe UI Emoji","Noto Color Emoji",sans-serif`;ctx.textAlign='center';ctx.textBaseline='middle';ctx.fillText(emojis[kind]?.[level]||'🏢',(gx+spec.width/2)*cell,(gy+spec.height/2)*cell);ctx.globalAlpha=1}}
if(hover&&!drag?.moved&&!(toolKind()==='road'||selectedSpec())){const [x,y]=hover,occupied=roads.has(key(x,y))||state.buildings.some(b=>x>=b.x&&x<b.x+b.width&&y>=b.y&&y<b.y+b.height)||!owned.has(plot(x,y));ctx.fillStyle=occupied?'#ffffff22':'#5fc27a55';ctx.fillRect(x*cell,y*cell,cell,cell)}ctx.restore();renderBuildMenu();updateTutorialFocus();
document.querySelector('#money').textContent=state.money;document.querySelector('#materials').textContent=state.materials;document.querySelector('#people').textContent=`${state.population} (мест ${state.capacity})`;document.querySelector('#jobs').textContent=`${state.employed} / ${state.jobs}`;document.querySelector('#labor-status').textContent=state.population>state.employed?`Безработица ${Math.round((state.population-state.employed)/state.population*100)}%`:state.jobs>state.employed?`Дефицит кадров ${Math.round((state.jobs-state.employed)/state.jobs*100)}%`:'Безработица 0%';document.querySelector('#happiness').textContent=state.happiness;document.querySelector('#time').textContent=state.remaining;
const info=document.querySelector('#info');if(state.message){info.textContent=state.message;return}if(hover){const [x,y]=hover,b=state.buildings.find(b=>b.kind!=='park'&&x>=b.x&&x<b.x+b.width&&y>=b.y&&y<b.y+b.height)||state.buildings.find(b=>b.kind==='park'&&x>=b.x&&x<b.x+b.width&&y>=b.y&&y<b.y+b.height);if(b){const ratio=b.workers/Math.max(1,b.limit);if(b.construction_ticks>0)info.textContent=`${b.title} — строительство, 1 игровой тик`;else if(b.fire_ticks>0)info.textContent=`${b.title} — пожар, остановка ещё ${b.fire_ticks} тик(а)`;else if(b.kind==='fire')info.textContent=`${b.title} (ур. ${b.level+1}) — пожарные ${b.workers}/${b.limit}, радиус ${b.fire_radius}${b.connected?'':' — нет дороги!'}${b.workers<b.limit?' — не хватает рабочих':''}`;else if(b.kind==='house')info.textContent=`${b.title} — вместимость ${b.limit}, счастье ${b.happiness}%${b.connected?'':' — нет дороги!'}${b.fire_protected?' — пожарная защита':''}`;else if(b.kind==='factory'||b.kind==='office')info.textContent=`${b.title} (ур. ${b.level+1}) — рабочие ${b.workers}/${b.limit}${b.connected&&ratio<.2?' — ПРОСТОЙ (<20%)':''}${b.connected?'':' — нет дороги!'}${b.fire_protected?' — пожарная защита':''}`;else info.textContent=`${b.title} (ур. ${b.level+1}) — ${b.bonus?`+${b.bonus}% счастья в радиусе ${b.radius}`:''}${b.connected?'':' — нет дороги!'}`;return}if(!owned.has(plot(x,y))){info.textContent=`Участок 10×10: клик по соседнему участку — купить за ${state.plot_cost}$`;return}}info.textContent=tool?'ЛКМ — действие, перетаскивание — двигать карту, ПКМ — отменить инструмент':'Обычный курсор — выберите инструмент строительства внизу'}
async function refresh(){try{const r=await fetch('/state');state=await r.json();draw()}catch(e){document.querySelector('#info').textContent='Соединение с игрой потеряно'}}
resize();refresh();setInterval(refresh,250);
</script></html>'''


class Handler(BaseHTTPRequestHandler):
    def log_message(self, _format: str, *args: object) -> None:
        pass

    def send_data(self, body: bytes, content_type: str = "application/json; charset=utf-8") -> None:
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self) -> None:
        if self.path == "/" or self.path.startswith("/?"):
            self.send_data(PAGE.encode("utf-8"), "text/html; charset=utf-8")
        elif self.path == "/state":
            self.send_data(json.dumps(GAME.snapshot(), ensure_ascii=False).encode("utf-8"))
        else:
            self.send_error(404)

    def do_POST(self) -> None:
        try:
            size = int(self.headers.get("Content-Length", "0"))
            data = json.loads(self.rfile.read(size) or b"{}")
            with GAME.lock:
                if self.path == "/act":
                    GAME.act(int(data["x"]), int(data["y"]), str(data["tool"]), bool(data.get("paint")))
                elif self.path == "/speed":
                    GAME.speed = max(1, min(60, int(data["speed"])))
                else:
                    self.send_error(404)
                    return
            self.send_data(b'{"ok":true}')
        except (KeyError, TypeError, ValueError, json.JSONDecodeError):
            self.send_error(400, "Invalid request")


def simulation_loop() -> None:
    last = time.monotonic()
    while True:
        time.sleep(.1)
        now = time.monotonic()
        with GAME.lock:
            GAME.elapsed += min(now - last, .5) * GAME.speed
            while GAME.elapsed >= 60:
                GAME.elapsed -= 60
                GAME.tick()
        last = now


def main() -> None:
    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
    address = f"http://127.0.0.1:{server.server_port}/"
    threading.Thread(target=simulation_loop, daemon=True).start()
    print(f"Игра открывается в браузере: {address}")
    print("Для выхода закройте это окно или нажмите Ctrl+C.")
    threading.Timer(.4, lambda: webbrowser.open(address)).start()
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nИгра остановлена.")
    finally:
        server.server_close()


if __name__ == "__main__":
    main()


Игра открывается в браузере: http://127.0.0.1:44661/
Для выхода закройте это окно или нажмите Ctrl+C.
